# 第12回: Planning and Reflection②

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session12/session12_planning_reflection_2.ipynb)

これまでのエージェントは、依頼を受けてから ReAct ループを回し、必要な Tool を呼び、答えを返してきた。この形は「聞かれたことに答える」までは強いが、「目的を達成する」には足りない。何を達成すれば終わりなのかが決まっておらず、途中の成果物が十分かを判断する仕組みも、うまく進んでいないときに軌道を変える仕組みもないためである。

今回は、エージェントを目的志向のシステムにする4つの能力を搭載する。

- **Goal Setting**: 何を達成すれば完了なのかを、観測できる条件として定める
- **Planning**: 目的へ至る手順を、実行可能なステップ列へ分解する
- **Reflection**: 出した成果物を自分で批評し、作り直す
- **Monitoring**: 実行結果を成功条件と照らし、続ける・作り直す・終える・人間へ戻すを決める

---

## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session12

%pip install -q -e "."

In [ ]:
# @title APIキーの設定
import getpass
import os
from pathlib import Path

# ローカル実行で .env がある場合はそこから読み込む
if Path('.env').exists():
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except ImportError:
        pass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

# 第5節で使う planning_agent.tools の web_search / fetch_url は、import 時に Tavily を初期化する
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('TAVILY_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')
print('Tavily APIキー設定完了' if os.environ.get('TAVILY_API_KEY') else 'Tavily APIキー未設定')

---

## 1. 目的志向のエージェント

複雑な依頼は、1回の応答や1回の Tool 呼び出しでは終わらない。まず何を達成すれば終わりかを定め、そこから中間ステップへ分解し、各ステップの成果物が十分かを判断し、全体が目的へ近づいているかを確かめる必要がある。

これらを1つの大きなプロンプトへ詰め込むこともできるが、そうすると「今どのステップにいるか」「何が未達か」「なぜやり直したか」がすべて会話文のコンテキストから認知・理解・判断をする必要がある。今回の実装では、目的・成功条件・計画・観測・判断を**状態**として持つことで、実行中に扱える制御情報へ変える。

責務を分けると、それぞれの役割は次のようになる。

| 能力 | 役割 | 状態として持つもの |
|---|---|---|
| Goal Setting | 何を達成すれば完了か | 目的、成功条件、制約 |
| Planning | どの順で何をするか | ステップ列、現在の位置 |
| Reflection | この成果物で十分か | 批評、指摘の履歴、改稿回数 |
| Monitoring | 全体として次に何をすべきか | 観測、未達ギャップ、次の遷移 |

Reflection と Monitoring は似ているが、見ている対象が違う。Reflection は**成果物の質**を見て、同じステップをやり直すかどうかを決める。Monitoring は**進行**を見て、次のステップへ進むか、計画を組み直すか、終わるか、人間へ戻すかを決める。この2つを1つのノードに混ぜると、「文章が硬い」といった品質の指摘と「計画が現実と合っていない」という進行の問題が同じ判断へ流れ込み、どちらの理由で止まったのかが追えなくなる。

### 統合ループ

4つの能力は、一度きりの直線的な処理ではない。計画は仮説であり、実行結果によって更新される。

[![](https://mermaid.ink/img/pako:eNplkc9L3EAUx_-VYU4KivQq4qmllwpFbxoP0-RtNjSZibMTFVyhm6jEpYdtoSy0WpWCPy5CoYqy4j8zJln_C99kd7NdvMybee99vt-ZNzvUlZ5D54mSEcwQGoAMWJmgO5a0OCEWVXUIwMKURR0mP1vUFHYpdoeMrwoRjHEpIrc-PkahwxS89ZgrmemqMb9h8uB4Ssgl4YAxsk2cGUY813yxZdeZVOTDsrkBIRI2ImioqTWL6uRcJw86vsX16fH4-fTBouvTZHZ2kbiC-djxHgNZAaU87i58knOLxa_r4ueeTnp52snaJ_nx2VMP8V6W3hb_9hAfmBi81Al9xlHnIwY-0tDxvU4OdJLopJulXd26y9KD_sWfijZQScM22Ei_wxApT_ASf_M_r-Pv2fVJ_-xrBRumhCXUfLAV8suD3UgAr57_PioOrxDOD-_7Vz8qeMgYvilh02tAsxR8XWW2DaFqkkBwM390WRrsRq-cGFDrsti_yDppZTTESilbcJxvNGE1WQ9CHxTWHcHBfFx-9KW4ibPTv9m3tvkyuvsCw-AEWg?type=png)](https://mermaid.live/edit#pako:eNplkc9L3EAUx_-VYU4KivQq4qmllwpFbxoP0-RtNjSZibMTFVyhm6jEpYdtoSy0WpWCPy5CoYqy4j8zJln_C99kd7NdvMybee99vt-ZNzvUlZ5D54mSEcwQGoAMWJmgO5a0OCEWVXUIwMKURR0mP1vUFHYpdoeMrwoRjHEpIrc-PkahwxS89ZgrmemqMb9h8uB4Ssgl4YAxsk2cGUY813yxZdeZVOTDsrkBIRI2ImioqTWL6uRcJw86vsX16fH4-fTBouvTZHZ2kbiC-djxHgNZAaU87i58knOLxa_r4ueeTnp52snaJ_nx2VMP8V6W3hb_9hAfmBi81Al9xlHnIwY-0tDxvU4OdJLopJulXd26y9KD_sWfijZQScM22Ei_wxApT_ASf_M_r-Pv2fVJ_-xrBRumhCXUfLAV8suD3UgAr57_PioOrxDOD-_7Vz8qeMgYvilh02tAsxR8XWW2DaFqkkBwM390WRrsRq-cGFDrsti_yDppZTTESilbcJxvNGE1WQ9CHxTWHcHBfFx-9KW4ibPTv9m3tvkyuvsCw-AEWg)

Monitoring は単なるログ記録ではなく、観測した結果をもとに、次の判断をする**制御ノード**。

- **complete**: 成功条件を満たしたので終了する
- **continue**: まだ作業が残っているので次のステップへ進む

実運用の Monitoring には、計画を組み直す replan と、人間へ判断を戻す escalate も置く。ここでは制御の骨格を見せることを優先し、この2つは扱わない（第6節で補足する）。

このノートブックでは、第2節で Session 11 から引き継いだ Reflection の内側ループを確認し、第3節で Goal Setting、第4節で Planning、第5節で Monitoring を足して外側ループを閉じる。実行部には、第9回と第10回で扱った Human-in-the-Loop と Sandbox を備えたエージェントを置く。

### 適用判断

動的な計画は万能ではない。柔軟性と予測可能性はトレードオフの関係にあり、手順が既に分かっている処理では、エージェントに計画させるより固定ワークフローの方が速く、安く、結果も安定する。請求書の定型チェック、決まったデータ変換、既定の承認フローなどがこれにあたる。

判断の軸は単純で、**how を実行中に発見する必要があるか**である。

| 判断基準 | 向く設計 |
|---|---|
| 手順が既知で、毎回同じ順序でよい | 固定ワークフロー |
| how を実行中に発見する必要がある | Planning + Monitoring |
| 成功条件を観測できる | Goal Setting + Monitoring |
| 出力の品質が速度やコストより重要 | Reflection |
| 失敗時の影響が大きい、判断が曖昧 | Human-in-the-Loop |
| 生成されたコマンドやコードを実行する | Sandbox |

Reflection にも同じトレードオフがある。改稿のたびに LLM 呼び出しが増えるため、コストとレイテンシは反復回数に比例して増え、履歴が伸びてコンテキストを圧迫する。品質が速度より重要な場合に限って使う。

---

## 2. Reflection

Reflection は、エージェントが自分の出力を評価し、その評価をもとに改善する設計パターン。単純な連鎖では出力がそのまま次の工程へ流れるが、Reflection は途中にフィードバックループを入れる。

処理は4段階になる。

1. **実行**: 最初の成果物を作る
2. **評価・批評**: 成果物を基準に照らして分析する
3. **改善**: 批評をもとに作り直す
4. **反復**: 満足のいく結果になるか、停止条件に達するまで繰り返す

重要なのは、評価する主体を生成する主体から**論理的に分離する**ことである。同じ役割のまま「見直して」と頼んでも、モデルは自分の出力を妥当だと見なしやすい。プロンプトを分け、別のペルソナ（例: 厳密なコードレビュアー、事実確認担当）を与えると、指摘は具体的になり、見落としも減る。この構成を Producer-Critic と呼ぶ。

[![](https://mermaid.ink/img/pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg?type=png)](https://mermaid.live/edit#pako:eNplkc9KAzEQxl8l5KRg9V6kFz0qiEeNSNxN29DuZslmVWh72AXRYg-K_6FaFFGrqAdBaD34MHGzfQyTVFvQnGYmv2_yTaYGHeYSmAewWGVbThlzARaWkQ_0ETisTKwiKJNPmfRk8irjh_QoHrw9pp2-6p8iuDYJcrkCCDhzI4dwzS79hLMbfKaQHXfU3oGML2Syr-Fh11_YKl2Oi0LLNKau2lmzO8LsjWUcTgV1NDRnA9t5aEDGT9nOvUyaMj6T8Z1q9gbdkz_PDdWmUR1BTjZpSPJAtXbV4bkeRwfZbR_B-sjWPxV2HBKIOmCRMJ-RvrS0WTP6kKxSj5oJ0ritnm_U5fXXx7u16OHtdSoIx4IyP9QCkJseTYN8OAVgiVNXf73gEdGZR7iHbQHWkPWBoCgTjyBdQtDFvIKguWgYbYD9Fca8sZyzqFQep1HgYkHmKS5xbKgiroamTlwqGF_8WbpdfuMbKtnPrg)

Reflection が効くのは、品質・正確さ・制約への適合が速度やコストより重要な場面である。

| 領域 | 生成するもの | 批評の観点 |
|---|---|---|
| 文章生成 | 記事、告知文、要約 | 流れ、トーン、明確さ、抜けている論点 |
| コード生成 | 関数、スクリプト | バグ、境界条件、テスト結果、可読性 |
| 情報統合 | 長文の要約 | 原文の要点との照合、事実の取りこぼし |
| 計画立案 | 手順、戦略 | 実行可能性、制約違反、依存関係の矛盾 |

### 2.1 Reflectionの実装

階乗を計算する Python 関数の実装

- Critic には**品質チェックリスト**を明示的に渡し、その各項目に照らして判定させる。チェックリストは第3節以降の「成功条件」に相当するもので、Reflection が何を基準に合否を出すのかを外から決めるための仕組みを提供する。
- Critic の戻り値は文章ではなく構造化出力にする。`verdict` が制御に使え、`issues` がそのまま次の改稿の入力になる。
- Critic には、未達項目のうち**最も重要なものを1件だけ**挙げさせる。一度に全部直させるより、どの指摘が成果物のどこを変えたかを追いやすく、レビューの実務にも近い。

In [ ]:
# @title Reflection ループの実装
from typing import Literal

from IPython.display import Image, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.runnables.graph_mermaid import CurveStyle
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict

model_id = 'gpt-5.4-nano' # @param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
llm = ChatOpenAI(model=model_id, temperature=0.2)

TASK_PROMPT = """`calculate_factorial` という名前の Python 関数を書いてください。
整数 `n` を受け取り、その階乗 (n!) を返します。
"""

# 批評の判定基準。第3節以降の「成功条件」にあたる
QUALITY_CHECKLIST = [
    'docstring に引数、戻り値、送出する例外が書かれている',
    '0 の階乗が 1 になる',
    '負の数には ValueError を送出する',
    'bool は int のサブクラスなので、True / False を拒否する',
    '再帰ではなく反復で実装し、大きな n でも再帰上限に達しない',
    '使用例が doctest 形式で書かれている',
]

PRODUCER_SYSTEM = 'あなたは Python の実装者です。要求を満たすコードだけを返します。説明文は書きません。'

CRITIC_SYSTEM = """あなたは Python に詳しいシニアソフトウェアエンジニアです。
提示されたコードを品質チェックリストの各項目に照らしてレビューしてください。
未達の項目があれば、そのうち最も重要なものを1件だけ issues に入れ、verdict を revise にします。
すべて満たしていれば verdict を accept とし、issues は空にします。"""


class Critique(BaseModel):
    """批評の結果。verdict がループの制御に、issues が次の改稿の入力になる。"""

    verdict: Literal['accept', 'revise'] = Field(
        description='全項目を満たすなら accept、そうでなければ revise。'
    )
    issues: list[str] = Field(description='最も重要な未達項目を1件だけ。accept のときは空。')
    guidance: str = Field(description='次の改稿で何をすべきかの指示。1文。')


# Critic だけ構造化出力にする。Producer の出力は成果物そのものなので素の文字列でよい
critic = llm.with_structured_output(Critique)


def checklist_text() -> str:
    return '\n'.join(f'- {item}' for item in QUALITY_CHECKLIST)


# ここから反復。何回目の改稿か、これまでどんな指摘を受けたか、いつ止めるか。
# これらを関数の引数で持ち回るとノードの責務がすぐ曖昧になるため、グラフの状態として持つ
class ReflectionState(TypedDict, total=False):
    task: str
    draft: str
    critique: Critique
    revision_notes: list[str]  # これまでに受けたすべての指摘（記憶）
    issue_log: list[list[str]]  # サイクルごとの指摘。比較のために残す
    iteration: int
    max_iterations: int
    use_memory: bool


def generate(state: ReflectionState) -> dict:
    """初回は生成、2回目以降は指摘を反映して書き直す。"""
    messages = [SystemMessage(content=PRODUCER_SYSTEM), HumanMessage(content=state['task'])]

    if state.get('draft'):
        if state.get('use_memory', True):
            # 記憶あり: 前回の成果物と、これまでの指摘をすべて渡す
            context = f'前回の成果物:\n{state["draft"]}\n\n'
            context += 'これまでに受けた指摘:\n' + '\n'.join(
                f'- {note}' for note in state.get('revision_notes', [])
            )
        else:
            # 記憶なし: 直近の指摘だけを渡す。前回の成果物も過去の指摘も見えない
            context = '次の指摘を踏まえて書いてください。\n' + '\n'.join(
                f'- {issue}' for issue in state['critique'].issues
            )
        messages.append(HumanMessage(content=context))

    response = llm.invoke(messages)
    return {'draft': str(response.content), 'iteration': state.get('iteration', 0) + 1}


def critique(state: ReflectionState) -> dict:
    """成果物をチェックリストに照らして批評し、指摘を記憶へ積む。"""
    result = critic.invoke([
        SystemMessage(content=CRITIC_SYSTEM),
        HumanMessage(content=(
            f'元の要求:\n{TASK_PROMPT}\n\n'
            f'品質チェックリスト:\n{checklist_text()}\n\n'
            f'レビュー対象:\n{state['draft']}'
        )),
    ])
    return {
        'critique': result,
        'revision_notes': state.get('revision_notes', []) + result.issues,
        'issue_log': state.get('issue_log', []) + [result.issues],
    }


def route_after_critique(state: ReflectionState) -> Literal['generate', 'finish']:
    """批評の verdict を分岐に使う。書き直すか、ここで終わるか。"""
    if state['critique'].verdict == 'accept':
        return 'finish'
    # 上限は付け足しの安全装置ではなく必須の構成要素。批評する側は「もっと良くできる点」を
    # いくらでも挙げられるので、accept が出るまで回す設計にすると止まらない（後の実行で確認する）
    if state.get('iteration', 0) >= state.get('max_iterations', 3):
        return 'finish'
    return 'generate'


reflection_builder = StateGraph(ReflectionState)
reflection_builder.add_node('generate', generate)
reflection_builder.add_node('critique', critique)

reflection_builder.add_edge(START, 'generate')
reflection_builder.add_edge('generate', 'critique')
reflection_builder.add_conditional_edges(
    'critique',
    route_after_critique,
    {'generate': 'generate', 'finish': END},
)

reflection_graph = reflection_builder.compile()

print('Model:', model_id)
display(Image(reflection_graph.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)))

In [ ]:
# @title 反復ループを実行する
def run_reflection(*, use_memory: bool, max_iterations: int = 3) -> ReflectionState:
    return reflection_graph.invoke(
        {
            'task': TASK_PROMPT,
            'max_iterations': max_iterations,
            'use_memory': use_memory,
            'revision_notes': [],
            'issue_log': [],
        },
        config={'recursion_limit': 30},
    )


with_memory = run_reflection(use_memory=True)

print(f'改稿回数: {with_memory["iteration"]} / 最終判定: {with_memory["critique"].verdict}')
for index, issues in enumerate(with_memory['issue_log'], 1):
    print(f'\n--- {index} 回目の批評 ({len(issues)} 件) ---')
    for issue in issues:
        print('-', issue[:120])

display(Markdown(f'#### 最終成果物\n{with_memory["draft"]}'))

### 注意点

- **批評は尽きない**: 上の実行で見たとおり、Critic は要求を満たした後も「より厳密にできる点」を挙げ続ける。`accept` を待つ設計にすると止まらないため、`max_iterations` が必須になる
- **コストとレイテンシが反復に比例する**: 1サイクルごとに生成と批評で2回の LLM 呼び出しが増える。履歴も伸びるため、コンテキスト長の上限にも近づく
- **自己評価は万能ではない**: 同じモデルが書き手と評価者を兼ねると、見落としも共有される。静的チェックのように機械的に確かめられる観点は、LLM に判定させず自分で確かめる
- **止め方を決める**: 上限に達して終わった場合と、`accept` で終わった場合は意味が違う。どちらで終わったかを状態に残さないと、後段が品質を過信する

---

## 3. Goal Setting

Goal Setting は、ユーザーの依頼を「何を達成すれば完了か」という目標へ変換する能力。依頼文から、達成すべき最終状態である**目的**、完了判定に使う**成功条件**、守るべき**制約**を取り出す。

目標は、それを測る手段とセットで初めて意味を持つ。目的地を決めても、現在地が分からなければ到着したかを判断できない。後段の Monitoring が実行結果と成功条件を照合するため、成功条件は成果物や Tool の出力から観測できる粒度で定める。

目標の書き方としては SMART（具体的、測定可能、達成可能、関連性がある、期限や停止条件がある）が目安になる。評価する側が「完了」「未完了」「要修正」を判断できる程度に成功条件が明示されていることが重要。

In [ ]:
# @title Goal Setting の LCEL チェーン
from langchain_core.prompts import ChatPromptTemplate


class GoalSpec(BaseModel):
    """ユーザー依頼から取り出した目的と成功条件。"""

    objective: str = Field(description='達成すべき最終状態。1文で具体的に書く。')
    success_criteria: list[str] = Field(
        description='完了判定に使う、観測可能な成功条件。3から4件。'
    )
    constraints: list[str] = Field(description='守るべき制約や禁止事項。')


goal_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'ユーザー依頼を、達成すべき目的、観測可能な成功条件、制約へ分解する。\n'
        '成功条件は、成果物を読めば満たしているか判断できる粒度で書く。\n'
        '文字数や文の数のように、数えないと判定できない条件は成功条件に入れない。'
    )),
    ('human', '{request}'),
])

goal_chain = goal_prompt | llm.with_structured_output(GoalSpec)


def bullets(items) -> str:
    return '\n'.join(f'- {item}' for item in items) if items else '- なし'

In [ ]:
# @title Goal Setting を実行する
WRITING_REQUEST = (
    'AI エージェントに shell 実行を許すときの注意点を、社内共有用に3つの見出しでまとめてください。'
    '各見出しでは、どんなリスクがあるかと、それをどう抑えるかを書いてください。'
)

goal_result = goal_chain.invoke({'request': WRITING_REQUEST})

In [ ]:
# @title Goal Setting の結果

display(Markdown(
    f'**Objective**\n\n{goal_result.objective}\n\n'
    f'**Success Criteria**\n\n{bullets(goal_result.success_criteria)}\n\n'
    f'**Constraints**\n\n{bullets(goal_result.constraints)}'
))

### 注意点

成功条件の質が低いと、後段の計画と監視も安定しない。「安全に実行する」のような抽象的な条件は、何を観測すれば満たしたと判断できるかまで具体化する。

---

## 4. Planning

Planning は、高レベルの目標を実行可能なステップ列へ変換する能力。エージェントは第3節で定めた目的・成功条件・制約を受け取り、現在の状態から目標状態へ至る手順を作る。計画は事前に存在せず、依頼に応じて生成される。

### 計画は作り直せる

計画どおりに進まないとき、制約を計画へ追加し、残りの候補を評価し直して次の手順を組み直す。ただし第1節で触れたとおり、この柔軟性は予測可能性とトレードオフになる。実行のたびに手順が変わるため、同じ依頼でも同じ経路をたどるとは限らない。

このノートブックの実装は計画を1度だけ作る。作り直しの設計は第6節で扱う。

### 計画を文章ではなく状態として持つ

「まず計画を立ててから作業して」とプロンプトへ書くだけでも、モデルは手順らしきものを出力する。ただしそれは応答本文の一部であり、後から参照するには本文を読み直して解釈する必要がある。


### 4.1 Planning の実装

`planning_chain` は、第3節で得た目的・成功条件・制約を受け取り、実行順のステップ列を作る。各ステップには中間成果物を1つだけ割り当てる。1ステップ目で成果物全体を作らせず、後段の Monitoring が実行結果を段階的に観測できる粒度にするためである。

計画は `WorkPlan` の構造化出力として受け取る。Goal Setting と同様に分岐もループもないため、`planning_prompt | llm.with_structured_output(WorkPlan)` という LCEL チェーンだけで表現できる。

In [ ]:
# @title Planning の LCEL チェーン
class WorkPlan(BaseModel):
    """目的を達成するための実行計画。"""

    steps: list[str] = Field(description='実行順のステップ。3件以内。各ステップは中間成果物を1つだけ作る。')


planning_prompt = ChatPromptTemplate.from_messages([
    ('system', (
        '目的を達成するための実行計画を作る。'
        '各ステップは、1つの中間成果物だけを作る大きさにする。'
    )),
    ('human', (
        'Objective:\n{objective}\n\n'
        'Success Criteria:\n{success_criteria}\n\n'
        'Constraints:\n{constraints}'
    )),
])

planning_chain = planning_prompt | llm.with_structured_output(WorkPlan)


plan_result = planning_chain.invoke({
    'objective': goal_result.objective,
    'success_criteria': bullets(goal_result.success_criteria),
    'constraints': bullets(goal_result.constraints),
})


def numbered(items) -> str:
    return '\n'.join(f'{n}. {item}' for n, item in enumerate(items, 1)) if items else 'なし'


display(Markdown(
    f'**Plan**\n\n{numbered(plan_result.steps)}'
))

---

## 5. Monitoring

Monitoring は、実行結果・環境の状態・Tool の出力を観測し、第3節で定めた成功条件と照らして次の一手を決める仕組みである。第4節の計画は仮説であり、実行結果によって更新される。ここで初めて Goal Setting・Planning・Reflection・Monitoring の4つを状態遷移として統合する。

[![](https://mermaid.ink/img/pako:eNplkctKw0AUhl9lmHWL-yJdiFt3rnRKGZOTNJhkwslExbZgGpFKXYgI4gWlG0GRLBREN77M0LR9C2dioFVX5zLnm_8_M13qomfTBpGYQI3QADDgZYN2GbKQEEZlBwJgusWozXGXUXPQp3o64uGWEMECR5G4nUWZRDaXsO5xF7mZcrgfmz7YnhS4IWwwQpaJtSrq2vHFvtXhKMnmmnFACByAtc2oCYmEdiwhWt3BlaYafKrsRGWZyq7U4GKSP8zGZ4y2SL3eJAiOD5bUXJUtuGJ4XtzfTU-fVJrP3p41OnscqcFIoz-CFWHu6SHseTH0ShP_T7llQSR7JBCh2UmrVVk7QuEixHGpOD961daMx-H75Gus0utluQopL2T6HULphQk0SPGiJ_NfW6YfjC57-YsGkQ9So9PbfHpzPE8v9a6GcLyQ-9pdGb1D_Z8t2v8GU1vNDw?type=png)](https://mermaid.live/edit#pako:eNplkctKw0AUhl9lmHWL-yJdiFt3rnRKGZOTNJhkwslExbZgGpFKXYgI4gWlG0GRLBREN77M0LR9C2dioFVX5zLnm_8_M13qomfTBpGYQI3QADDgZYN2GbKQEEZlBwJgusWozXGXUXPQp3o64uGWEMECR5G4nUWZRDaXsO5xF7mZcrgfmz7YnhS4IWwwQpaJtSrq2vHFvtXhKMnmmnFACByAtc2oCYmEdiwhWt3BlaYafKrsRGWZyq7U4GKSP8zGZ4y2SL3eJAiOD5bUXJUtuGJ4XtzfTU-fVJrP3p41OnscqcFIoz-CFWHu6SHseTH0ShP_T7llQSR7JBCh2UmrVVk7QuEixHGpOD961daMx-H75Gus0utluQopL2T6HULphQk0SPGiJ_NfW6YfjC57-YsGkQ9So9PbfHpzPE8v9a6GcLyQ-9pdGb1D_Z8t2v8GU1vNDw)

### Reflection と Monitoring を分ける

第2節で作った Reflection は、成果物の質を見て同じ作業をやり直すループだった。Monitoring はその外側で、進行そのものを制御する。

| | Reflection | Monitoring |
|---|---|---|
| 見る対象 | 直近のステップの成果物 | 目的全体に対する進捗 |
| 判断 | このまま採用するか、やり直すか | 続けるか、終えるか |
| 遷移先 | 同じステップの再実行 | 次のステップ、完了 |
| 上限 | `max_reflections` | `max_steps` |


### 5.1 統合グラフとしてMonitoringを実装

In [ ]:
# @title 統合グラフの状態とノード
from planning_agent.state import Context, LongTermMemoryState

NextAction = Literal['continue', 'complete']


class PlannerState(LongTermMemoryState, total=False):
    """4つの能力が共有する統合グラフの状態。"""

    # Goal Setting: 何を達成すれば完了か
    user_request: str  # ユーザーから受け取った元の依頼
    objective: str  # 依頼から定めた達成目的
    success_criteria: list[str]  # 完了を判定するための観測可能な条件
    constraints: list[str]  # 実行中に守るべき制約

    # Planning: どの順で何をするか
    plan: list[str]  # 目的を達成するための実行ステップ列
    current_step_index: int  # 現在実行しているステップの位置
    briefed: bool  # 目的・成功条件・制約を実行部へ渡し済みか

    # Reflection: 直近の成果物で十分か
    critique: str  # 直近の成果物に対する批評と改稿指示
    revision_notes: list[str]  # これまでの批評で指摘された問題の履歴
    reflection_count: int  # 現在までに改稿した回数
    max_reflections: int  # 許可する改稿回数の上限

    # Monitoring: 全体として次に何をすべきか
    completed_steps: list[str]  # 完了として記録したステップ
    observations: list[str]  # 各ステップの実行結果から得た観測の履歴
    last_result: str  # 直近のステップが生成した成果物
    monitor_reason: str  # Monitoring が次の行動を選んだ理由
    remaining_gaps: list[str]  # まだ未達または未検証の成功条件
    next_action: NextAction  # Monitoring が選んだ次の遷移
    max_steps: int  # 実行できる全ステップ数の上限

    # 出口
    final_answer: str  # ユーザーへ返す最終回答


def current_step(state: PlannerState) -> str:
    index = state.get('current_step_index', 0)
    plan = state.get('plan', [])
    return plan[index] if index < len(plan) else '追加で実行するステップはない。'


def goal_setting(state: PlannerState) -> dict:
    """Goal Setting の LCEL チェーンをグラフの状態更新へ変換する。"""
    request = state.get('user_request') or str(state['messages'][-1].content)
    result = goal_chain.invoke({'request': request})
    return {
        'user_request': request,
        'objective': result.objective,
        'success_criteria': result.success_criteria,
        'constraints': result.constraints,
    }


def plan_task(state: PlannerState) -> dict:
    """Planning の LCEL チェーンをグラフの状態更新へ変換する。"""
    result = planning_chain.invoke({
        'objective': state['objective'],
        'success_criteria': bullets(state['success_criteria']),
        'constraints': bullets(state['constraints']),
    })
    return {'plan': result.steps, 'current_step_index': 0}


def initialize_control(state: PlannerState) -> dict:
    """Reflection と Monitoring の制御状態を初期化する。"""
    return {
        'briefed': False,
        'completed_steps': [],
        'observations': [],
        'last_result': '',
        'critique': '',
        'revision_notes': [],
        'reflection_count': 0,
        'monitor_reason': '',
        'remaining_gaps': [],
        'next_action': 'continue',
        'final_answer': '',
    }


def last_ai_text(state: PlannerState) -> str:
    for message in reversed(state.get('messages', [])):
        if isinstance(message, AIMessage) and message.content:
            return str(message.content)
    return '最終応答を取得できなかった。'


class StepCritique(BaseModel):
    """ステップの成果物に対する批評。"""

    verdict: Literal['accept', 'revise'] = Field(
        description='ステップの目的を満たしていれば accept、やり直すべきなら revise。'
    )
    issues: list[str] = Field(description='不足している点。accept のときは空。')
    guidance: str = Field(description='やり直す場合の指示。1文。')


step_critic = llm.with_structured_output(StepCritique)


def execute_step(state: PlannerState) -> dict:
    """現在のステップを指示文へ組み立て、実行部へ渡す。LLM は呼ばない。"""
    # messages は追記され続けるため、実行中に変わらない目的・成功条件・制約を毎回積むと、
    # 同じ内容が履歴に複製される。初回だけ渡し、2回目以降はステップだけを渡す
    if state.get('briefed'):
        instruction = f"""Current Step:
{current_step(state)}
"""
    else:
        instruction = f"""あなたは大きなタスクの1ステップだけを担当する実行担当です。

Objective:
{state['objective']}

Success Criteria:
{bullets(state['success_criteria'])}

Constraints:
{bullets(state['constraints'])}

Current Step:
{current_step(state)}
"""
    if state.get('critique'):
        instruction += (
            f'\n前回の成果物への指摘:\n{state["critique"]}\n'
            'この指摘を反映して、同じステップをやり直してください。\n'
        )
    instruction += '\nこのステップの成果物と、残った不確実性を報告してください。'
    return {'messages': [HumanMessage(content=instruction)], 'briefed': True}


def reflect_step(state: PlannerState) -> dict:
    """ステップの成果物を批評する。revise なら同じステップをやり直す。"""
    # 長期記憶があれば批評の材料に加える。ユーザーの好みや制約は
    # 成功条件に書かれていないことが多く、批評の側で効いてくる
    memories = state.get('memories', [])
    memory_note = f'\n\nユーザーについて分かっていること:\n{bullets(memories)}' if memories else ''

    result = step_critic.invoke([
        SystemMessage(content=(
            'ステップの成果物を、そのステップの目的と成功条件に照らして批評する。'
            '根拠のない accept は禁止。指摘は1件に絞る。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state["objective"]}\n\n'
            f'Success Criteria:\n{bullets(state["success_criteria"])}\n\n'
            f'Current Step:\n{current_step(state)}\n\n'
            f'過去に受けた指摘:\n{bullets(state.get("revision_notes", []))}'
            f'{memory_note}\n\n'
            f'成果物:\n{state["last_result"]}'
        )),
    ])
    if result.verdict == 'accept':
        return {'critique': ''}
    return {
        'critique': '\n'.join(result.issues + [result.guidance]),
        'revision_notes': state.get('revision_notes', []) + result.issues,
        'reflection_count': state.get('reflection_count', 0) + 1,
    }


def collect_step_result(state: PlannerState) -> dict:
    """実行部の最後の応答を、このステップの成果物として取り出す。"""
    return {'last_result': last_ai_text(state)}


def route_after_reflect(state: PlannerState) -> Literal['execute_step', 'monitor_progress']:
    # 指摘があり、改稿の上限に達していなければ同じステップをやり直す
    if state.get('critique') and state.get('reflection_count', 0) <= state.get('max_reflections', 1):
        return 'execute_step'
    return 'monitor_progress'

In [ ]:
# @title monitor_progress / finalize
from langgraph.types import Command


class MonitorDecision(BaseModel):
    """観測結果に対する制御判断。"""

    next_action: NextAction = Field(
        description='次のステップへ進む continue、目的を達成した complete のいずれか。'
    )
    remaining_gaps: list[str] = Field(description='まだ満たしていない成功条件。')
    reason: str = Field(description='判断理由。観測結果に基づいて簡潔に書く。')


monitor = llm.with_structured_output(MonitorDecision)


def monitor_progress(state: PlannerState) -> dict:
    """ステップを記録し、成功条件と照らして次の遷移を決める。"""
    step_number = state.get('current_step_index', 0) + 1
    completed = state.get('completed_steps', []) + [f'{step_number}. {current_step(state)}']
    observations = state.get('observations', []) + [state['last_result']]

    # ステップを1つ進め、Reflection のカウンタを次のステップ用に戻す
    progress = {
        'completed_steps': completed,
        'observations': observations,
        'current_step_index': step_number,
        'reflection_count': 0,
        'critique': '',
    }

    # 上限判定は LLM に委ねない
    if len(observations) >= state.get('max_steps', 4):
        return {
            **progress,
            'next_action': 'complete',
            'monitor_reason': '最大ステップ数に到達したため、ここで打ち切る。',
            'remaining_gaps': state.get('success_criteria', []),
        }

    decision = monitor.invoke([
        SystemMessage(content=(
            '実行結果を成功条件に照らして監視し、次の制御を選ぶ。\n'
            '- 成功条件を満たしていれば complete\n'
            '- 未達が残っていても、計画の残りステップで解消できるなら continue\n'
            '観測から判断できない条件は、未達ではなく未検証として扱う。'
            '未検証の条件は remaining_gaps に挙げたうえで continue を選ぶ。\n'
            '根拠の弱い complete は禁止する。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state["objective"]}\n\n'
            f'Success Criteria:\n{bullets(state["success_criteria"])}\n\n'
            f'Plan:\n{numbered(state["plan"])}\n\n'
            f'Completed Steps:\n{bullets(completed)}\n\n'
            f'Latest Observation:\n{state["last_result"]}'
        )),
    ])
    return {
        **progress,
        'next_action': decision.next_action,
        'monitor_reason': decision.reason,
        'remaining_gaps': decision.remaining_gaps,
    }


def finalize(state: PlannerState) -> dict:
    """完了理由、観測、残リスクをまとめて最終報告にする。"""
    response = llm.invoke([
        SystemMessage(content=(
            'Planner の最終報告を日本語で簡潔に作る。'
            '完了した成果物、根拠、残っているリスクを分けて書く。'
        )),
        HumanMessage(content=(
            f'Objective:\n{state.get("objective")}\n\n'
            f'Completed Steps:\n{bullets(state.get("completed_steps", []))}\n\n'
            f'Observations:\n{bullets(state.get("observations", []))}\n\n'
            f'Remaining Gaps:\n{bullets(state.get("remaining_gaps", []))}\n\n'
            f'Monitoring の判断:\n{state.get("monitor_reason", "")}'
        )),
    ])
    return {'final_answer': str(response.content),
            'messages': [AIMessage(content=str(response.content))]}

In [ ]:
# @title ルーティングとグラフ構築
from planning_agent.memory import (
    load_memory,
    make_extract_memory,
    validate_memory,
    write_memory,
)
from langgraph.checkpoint.memory import InMemorySaver


def route_after_monitor(state: PlannerState) -> Literal['execute_step', 'finalize']:
    if state.get('next_action', 'continue') == 'complete':
        return 'finalize'
    # 継続でも、計画を使い切っていれば終わる
    if state.get('current_step_index', 0) >= len(state.get('plan', [])):
        return 'finalize'
    return 'execute_step'


def build_planner_graph(executor) -> StateGraph:
    """計画ループを組み立てる。executor だけが差し替え可能な実行部。

    第5回の長期記憶ノードをループの前後へ置く。目的設定より前に記憶を読み、
    最終報告のあとで保存する。
    """
    builder = StateGraph(PlannerState, context_schema=Context)

    builder.add_node('goal_setting', goal_setting)
    builder.add_node('plan_task', plan_task)
    builder.add_node('initialize_control', initialize_control)
    builder.add_node('execute_step', execute_step)
    builder.add_node('agent', executor)
    builder.add_node('collect_step_result', collect_step_result)
    builder.add_node('reflect_step', reflect_step)
    builder.add_node('monitor_progress', monitor_progress)
    builder.add_node('finalize', finalize)

    builder.add_node('load_memory', load_memory)
    builder.add_node('extract_memory', make_extract_memory(llm))
    builder.add_node('validate_memory', validate_memory)
    builder.add_node('write_memory', write_memory)
    builder.add_edge(START, 'load_memory')
    builder.add_edge('load_memory', 'goal_setting')
    builder.add_edge('finalize', 'extract_memory')
    builder.add_edge('extract_memory', 'validate_memory')
    builder.add_edge('validate_memory', 'write_memory')
    builder.add_edge('write_memory', END)

    builder.add_edge('goal_setting', 'plan_task')
    builder.add_edge('plan_task', 'initialize_control')
    builder.add_edge('initialize_control', 'execute_step')
    builder.add_edge('execute_step', 'agent')
    builder.add_edge('agent', 'collect_step_result')
    builder.add_edge('collect_step_result', 'reflect_step')
    builder.add_conditional_edges(
        'reflect_step',
        route_after_reflect,
        {'execute_step': 'execute_step', 'monitor_progress': 'monitor_progress'},
    )
    builder.add_conditional_edges(
        'monitor_progress',
        route_after_monitor,
        {'execute_step': 'execute_step', 'finalize': 'finalize'},
    )
    return builder

In [ ]:
# @title HITL と Sandbox を備えた実行部
# planning_agent.executor は第9回・第10回で組み立てた実行部を取り込んだもの。
# make_guarded_agent() は create_agent() の ReAct サブグラフに次の Middleware を載せて返す。
#   - 長期記憶とサンドボックス制約を system prompt へ差し込む dynamic_prompt
#   - shell Tool を提供する ShellToolMiddleware（ExecutionPolicy で隔離の強さを決める）
#   - shell / write_file / file_delete を承認対象にする HumanInTheLoopMiddleware
# ホストで直接動く run_command と python_repl は除外され、コマンド実行の入口は shell だけになる
from planning_agent.executor import make_guarded_agent
from planning_agent.hitl import SANDBOX_HITL_INTERRUPT_ON
from planning_agent.sandbox import make_shell_session_spec
from planning_agent.tools import DEFAULT_TOOLS, without_host_execution_tools

execution_policy = 'host' # @param ['host', 'docker', 'codex']

shell_spec = make_shell_session_spec(execution_policy)
print('承認が必要な Tool:', [name for name, config in SANDBOX_HITL_INTERRUPT_ON.items() if config])
print('明示的に渡す Tool:', [tool.name for tool in without_host_execution_tools(DEFAULT_TOOLS)])
print('shell Tool は ShellToolMiddleware が追加するため、上の一覧には現れない')
print(f'\n--- shell Tool の制約（{execution_policy}）---\n{shell_spec.note}')

guarded_executor = make_guarded_agent(
    llm,
    tools=DEFAULT_TOOLS,
    execution_policy=execution_policy,
)

In [ ]:
# @title 実行部を組み込んだグラフ
from langchain_openai import OpenAIEmbeddings
from langgraph.store.memory import InMemoryStore

long_term_store = InMemoryStore(
    index={
        'embed': OpenAIEmbeddings(model='text-embedding-3-small'),
        'dims': 1536,
        'fields': ['text'],
    }
)

planner_graph = build_planner_graph(guarded_executor).compile(
    checkpointer=InMemorySaver(),
    store=long_term_store,
)

# xray=2 で、agent ノードの中の ReAct サブグラフまで展開する
display(Image(
    planner_graph.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
))

### 5.3 ループを動かす

依頼には shell の実行を伴うものを使う。ReAct サブグラフがコマンドを組み立てた時点で `HumanInTheLoopMiddleware` が止めるため、最初の実行は承認待ちの中断で終わる。

In [ ]:
# @title 遷移を表示するヘルパー
import json
from uuid import uuid4

# 長期記憶の名前空間とモデルを決める実行時コンテキスト
planner_context = Context(user_id='session12-user', model=model_id)

# 遷移を追うために表示する状態。値が入った更新だけを出す
TRACE_KEYS = [
    'objective', 'plan', 'critique', 'last_result',
    'next_action', 'monitor_reason', 'remaining_gaps', 'final_answer',
]


def shorten(value, limit: int = 100) -> str:
    text = ' '.join(str(value).split())
    return text if len(text) <= limit else text[:limit] + '…'


def trace(graph, payload, config, context) -> None:
    """ノードの遷移と、主要な状態の変化だけを表示する。"""
    for chunk in graph.stream(payload, config=config, context=context, stream_mode='updates'):
        for node, update in chunk.items():
            if node == '__interrupt__':
                print('  ** 中断: Tool の承認待ち **')
                continue
            print(f'[{node}]')
            if not isinstance(update, dict):
                continue
            for key in TRACE_KEYS:
                if update.get(key):
                    print(f'    {key}: {shorten(update[key])}')

In [ ]:
# @title 実行例: shell の実行前に止まる
SHELL_REQUEST = (
    'shell Tool を使って 1 から 50 までの素数の個数を数え、'
    '使ったコマンドと結果を報告してください。'
)

planner_config = {
    'configurable': {'thread_id': f'planner-{uuid4()}'},
    'recursion_limit': 60,  # ループがあるため既定値では足りない
}

trace(
    planner_graph,
    {
        'messages': [HumanMessage(content=SHELL_REQUEST)],
        'user_request': SHELL_REQUEST,
        'max_steps': 3,
        'max_reflections': 1,
    },
    planner_config,
    planner_context,
)

review_request = planner_graph.get_state(planner_config).interrupts[0].value
print('\n--- 承認を求められた内容 ---')
print(json.dumps(review_request, ensure_ascii=False, indent=2)[:1000])

### 承認を返して再開する

中断の payload には `action_requests` が入っている。承認するには、要求と同じ件数の判断を並べた `{'decisions': [{'type': 'approve'}]}` を `Command(resume=...)` で返す。中断状態はチェックポインタにあるので、再開は同じ `thread_id` へ渡す。

ReAct ループは1ステップの中で何度も Tool を呼ぶため、承認要求は1回では終わらない。ここでは中断が起きるたびに承認を返す小さなループで、完了まで進める。

In [ ]:
# @title 承認しながら完了まで進める
def resume_until_done(graph, config, context, *, max_rounds: int = 10):
    """承認要求のたびに approve を返して、完了まで進める。

    実際のアプリケーションでは、ここが人間のレビュー画面にあたる。
    """
    for _ in range(max_rounds):
        snapshot = graph.get_state(config)
        if not snapshot.interrupts:
            return snapshot

        requests = snapshot.interrupts[0].value['action_requests']
        for request in requests:
            print(f'\n>> 承認: {request["name"]} {request["args"]}')

        resume = {'decisions': [{'type': 'approve'} for _ in requests]}
        trace(graph, Command(resume=resume), config, context)

    raise RuntimeError(f'中断が {max_rounds} 回を超えました')


completed = resume_until_done(planner_graph, planner_config, planner_context)
display(Markdown(f'#### 最終報告\n{completed.values["final_answer"]}'))